## SDSS Analysis Part 1

For the data analysis, I chose to look at the **Sloan Digital Sky Survey (SDSS)** database for the following reasons:



* It is one of the largest open-source databases of celestial objects, with over **500 million** objects identified through photometric data, and **5.8 million** with additional spectroscopic data.
* SDSS has a huge number of attributes, allowing for extensive analysis on many aspects of various objects in the night sky, such as stars, galaxies and quasars.
* It has easily-accessible data in CSV format through the...


The following SQL query was used to retrieve the CSV:

SELECT TOP 500000
p.objid,p.ra,p.dec,p.u,p.g,p.r,p.i,p.z,
p.run, p.rerun, p.camcol, p.field,
s.specobjid, s.class, s.z as redshift,
s.plate, s.mjd, s.fiberid, p.cx, p.cy, p.cz, p.type, p.flags

FROM PhotoObj AS p

JOIN SpecObj AS s ON s.bestobjid = p.objid

[Here is the Skyserver website from which you can download the SDSS database](https://skyserver.sdss.org/dr19)

![Screenshot of SQL query on SDSS Skyserver](sdss_sql_query.png)

Here is some info about the selected 50,000-item subset of the SDSS, followed by a table of the first 10 entries:

In [6]:
import pandas as pd

df = pd.read_csv('SDSS_500k_extended_v2.csv')

print(df.info())
print("""


""")
print(df.head(10))

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 23 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   objid      500000 non-null  int64  
 1   ra         500000 non-null  float64
 2   dec        500000 non-null  float64
 3   u          500000 non-null  float64
 4   g          500000 non-null  float64
 5   r          500000 non-null  float64
 6   i          500000 non-null  float64
 7   z          500000 non-null  float64
 8   run        500000 non-null  int64  
 9   rerun      500000 non-null  int64  
 10  camcol     500000 non-null  int64  
 11  field      500000 non-null  int64  
 12  specobjid  500000 non-null  uint64 
 13  class      500000 non-null  str    
 14  redshift   500000 non-null  float64
 15  plate      500000 non-null  int64  
 16  mjd        500000 non-null  int64  
 17  fiberid    500000 non-null  int64  
 18  cx         500000 non-null  float64
 19  cy         500000 non-null  float6

Firstly, I wanted to see the difference in classifications of objects between the `PhotoObj` and `SpecObj` data. In the `SDSS_50k_extended.csv` dataset, `class` is the spectroscopic classification, while `type` is the photometric identification.

Typically, the **spectroscopic** data is seen as the more accurate, "true classification", while the photometric data is just an educated guess.

To see the correlation between the photometric and spectroscopic classification, a crosstabulation can be created:

In [7]:
crossed = pd.crosstab(df["class"], df["type"])
print(crossed)

type     0       3       6
class                     
GALAXY  46  292443    9254
QSO     12    7282   73087
STAR    33    6794  111049


The `class` names displayed as a   `str` are relatively straightforward, with `QSO` being a **quasi-stellar object**, otherwise known as a **quasar**.

The `type` numbers displayed as an `int64` encode the following classifications:

0. Unknown object
1. Cosmic ray
2. Defect
3. Galaxy
4. Ghost
5. Known object
6. Star
7. Star trail
8. Sky

*From [https://www.sdss4.org/dr16/algorithms/classify/](https://www.sdss4.org/dr16/algorithms/classify/)*

In [8]:
print("GALAXIES:")
print("     Number correctly identified: "+str(len(df[(df["class"] == "GALAXY") & (df["type"] == 3)])))
percent_g = len(df[(df["class"] == "GALAXY") & (df["type"] == 3)])/len(df[df["class"]=="GALAXY"])*100
print(f"     Percentage correctly identified: {percent_g:.4f}%")
qso_as_g = len(df[(df["class"] == "QSO") & (df["type"] == 3)])
star_as_g = len(df[(df["class"] == "STAR") & (df["type"] == 3)])
print("\n     Most common misidentification of OTHER as GALAXY: "+("QSO" if qso_as_g > star_as_g else "STAR"))
g_as_un = len(df[(df["class"] == "GALAXY") & (df["type"] == 0)])
g_as_star = len(df[(df["class"] == "GALAXY") & (df["type"] == 6)])
print("     Most common misidentification of GALAXY as OTHER: "+("UNKNOWN" if g_as_un > g_as_star else "STAR"))

print("\n\n STARS:")
print("     Number correctly identified: "+str(len(df[(df["class"] == "STAR") & (df["type"] == 6)])))
percent_s = len(df[(df["class"] == "STAR") & (df["type"] == 6)])/len(df[df["class"]=="STAR"])*100
print(f"     Percentage correctly identified: {percent_s:.4f}%")
qso_as_s = len(df[(df["class"] == "QSO") & (df["type"] == 6)])
gal_as_s = len(df[(df["class"] == "GALAXY") & (df["type"] == 6)])
print("\n     Most common misidentification of OTHER as STAR: "+("QSO" if qso_as_s > gal_as_s else "GALAXY"))
s_as_un = len(df[(df["class"] == "STAR") & (df["type"] == 0)])
s_as_gal = len(df[(df["class"] == "STAR") & (df["type"] == 3)])
print("     Most common misidentification of STAR as OTHER: "+("UNKNOWN" if s_as_un > s_as_gal else "GALAXY"))

GALAXIES:
     Number correctly identified: 292443
     Percentage correctly identified: 96.9179%

     Most common misidentification of OTHER as GALAXY: QSO
     Most common misidentification of GALAXY as OTHER: STAR


 STARS:
     Number correctly identified: 111049
     Percentage correctly identified: 94.2083%

     Most common misidentification of OTHER as STAR: QSO
     Most common misidentification of STAR as OTHER: GALAXY


From these calculations, it is clear that photometric data is about 3% more accurate at identifying galaxies as stars.

Also, when the photometric result is wrong, it is much more likely to recognise a star as a galaxy or a galaxy as a star, rather than putting it down as `UNKNOWN`. This suggests that the **recognition model is overly confident**.

Furthermore, quasars (`QSO`) are 9.5x as likely to be identified as stars compared to galaxies when there is no `type` category specifically for quasars. This is unsurprising, since quasars are very compact, ultra-bright black hole accretion discs. This means that they emit heaps of light in a very small area (similar to a star). Also, quasars are found in the middle of galaxies, so they usually way outshine the host galaxy.